# SSL Embeddings (IEMOCAP)

This notebook extracts pooled self-supervised embeddings (HuBERT/wav2vec2).
Each utterance becomes one training row for downstream SER models.

In [17]:
# Install deps: uv add torch transformers protobuf soundfile tqdm
from pathlib import Path
import time
from concurrent.futures import ThreadPoolExecutor

import numpy as np
import pandas as pd
import soundfile as sf
from tqdm.auto import tqdm


In [18]:
# Configuration
REPO_ROOT = Path.cwd().parents[1]  # repo root (notebook is under feature_extraction/)
CSV_PATH = REPO_ROOT / "datasets" / "IEMOCAP" / "iemocap_full_dataset.csv"
AUDIO_ROOT = REPO_ROOT / "datasets" / "IEMOCAP"
OUT_DIR = REPO_ROOT / "extracted_features" / "ssl_embeddings"
OUT_FILE = "ssl_embeddings_features.csv"

# SSL params
MODEL_NAME = "facebook/hubert-base-ls960"
TARGET_SR = 16_000
DEVICE = None  # "cuda" or "cpu"; None selects automatically
POOL = ("mean", "std")
LAYER = 6  # None uses last hidden state

# Runtime params
BATCH_SIZE = None  # None = auto-tune based on available VRAM
MIN_BATCH_SIZE = 4
MAX_BATCH_SIZE = 48
FLUSH_EVERY = 10  # batches
GPU_PREFETCH_WORKERS = 4  # host-side decode workers for CUDA execution
USE_AMP = True  # mixed precision on CUDA

OUT_DIR.mkdir(parents=True, exist_ok=True)
OUT_PATH = OUT_DIR / OUT_FILE
OUT_PATH


PosixPath('/home/marcello/Speech-Emotion-Recognition/extracted_features/ssl_embeddings/ssl_embeddings_features.csv')

In [19]:
import os
from contextlib import nullcontext

_MODEL_CACHE: dict[str, tuple[object, object, str]] = {}
_TORCH_CONFIGURED = False


def _detect_device(device: str | None) -> str:
    torch, _ = _require_torch_stack()
    if device is not None:
        return device
    return "cuda" if torch.cuda.is_available() else "cpu"


def _auto_batch_size(device: str) -> int:
    if BATCH_SIZE is not None:
        return int(BATCH_SIZE)

    default_batch = 8
    if device != "cuda":
        return default_batch

    torch, _ = _require_torch_stack()
    try:
        total_vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    except Exception:
        return default_batch

    if total_vram_gb >= 20:
        return min(MAX_BATCH_SIZE, 32)
    if total_vram_gb >= 12:
        return min(MAX_BATCH_SIZE, 16)
    return min(MAX_BATCH_SIZE, default_batch)


def _gpu_prefetch_workers(device: str) -> int:
    if device != "cuda":
        return 1
    return max(1, min(GPU_PREFETCH_WORKERS, os.cpu_count() or 1))


def _load_audio_item(meta_row: pd.Series):
    rel_path = str(meta_row["path"])
    audio_path = AUDIO_ROOT / rel_path
    if not audio_path.exists():
        return None, str(audio_path), None

    try:
        audio, sr = sf.read(audio_path, dtype="float32", always_2d=False)
        if audio.ndim > 1:
            audio = audio.mean(axis=1)
        if sr != TARGET_SR:
            audio = _resample(audio, sr, TARGET_SR)
            sr = TARGET_SR
        duration_s = float(audio.shape[0] / sr)
        return (meta_row, duration_s, rel_path), None, audio
    except Exception as exc:
        return None, f"{audio_path} | load_error: {exc}", None


def _require_torch_stack():
    global _TORCH_CONFIGURED
    try:
        import torch
        from transformers import AutoModel, AutoFeatureExtractor
    except ImportError as exc:
        raise ImportError(
            "SSL embeddings require torch and transformers. "
            "Install with: uv add torch transformers protobuf"
        ) from exc

    if not _TORCH_CONFIGURED:
        num_threads = max(1, (os.cpu_count() or 1) - 2)
        os.environ.setdefault("OMP_NUM_THREADS", str(num_threads))
        os.environ.setdefault("MKL_NUM_THREADS", str(num_threads))
        torch.set_num_threads(num_threads)
        _TORCH_CONFIGURED = True

    return torch, (AutoModel, AutoFeatureExtractor)


def _get_model(model_name: str, device: str | None):
    torch, (AutoModel, AutoFeatureExtractor) = _require_torch_stack()

    device = _detect_device(device)

    cached = _MODEL_CACHE.get(model_name)
    if cached is None:
        fe = AutoFeatureExtractor.from_pretrained(model_name)
        model = AutoModel.from_pretrained(model_name)
        model.eval()
        model = model.to(device)
        _MODEL_CACHE[model_name] = (fe, model, device)
    else:
        fe, model, cached_device = cached
        if cached_device != device:
            model = model.to(device)
            _MODEL_CACHE[model_name] = (fe, model, device)

    return fe, model, device


def _resample(audio: np.ndarray, sr: int, target_sr: int) -> np.ndarray:
    # Resample audio to target_sr, preferring torchaudio if available
    if sr == target_sr:
        return audio.astype(np.float32, copy=False)

    try:
        import torch
        import torchaudio

        waveform = torch.from_numpy(audio.astype(np.float32, copy=False))
        if waveform.ndim == 1:
            waveform = waveform.unsqueeze(0)
        resampler = torchaudio.transforms.Resample(
            orig_freq=sr,
            new_freq=target_sr,
        )
        resampled = resampler(waveform).squeeze(0).cpu().numpy()
        return resampled.astype(np.float32, copy=False)
    except Exception:
        import librosa

        return librosa.resample(
            audio.astype(np.float32, copy=False),
            orig_sr=sr,
            target_sr=target_sr,
        ).astype(np.float32, copy=False)


def _get_feat_lengths(model, lengths):
    if hasattr(model, "_get_feat_extract_output_lengths"):
        return model._get_feat_extract_output_lengths(lengths)
    return lengths


def extract_ssl_embeddings_batch(
    audios: list[np.ndarray],
    sr: int,
    *,
    model_name: str,
    device: str | None,
    target_sr: int,
    pool: tuple[str, ...],
    layer: int | None,
) -> list[dict[str, np.ndarray]]:
    torch, _ = _require_torch_stack()
    fe, model, device = _get_model(model_name, device)

    if sr != target_sr:
        audios = [_resample(audio, sr, target_sr) for audio in audios]
        sr = target_sr

    inputs = fe(audios, sampling_rate=target_sr, return_tensors="pt", padding=True)
    input_values = inputs["input_values"].to(device, non_blocking=(device == "cuda"))
    attention_mask = inputs.get("attention_mask")
    if attention_mask is not None:
        attention_mask = attention_mask.to(device, non_blocking=(device == "cuda"))

    autocast_ctx = (
        torch.autocast(device_type="cuda", dtype=torch.float16)
        if device == "cuda" and USE_AMP
        else nullcontext()
    )

    with torch.no_grad():
        with autocast_ctx:
            if layer is None:
                outputs = model(input_values, attention_mask=attention_mask)
                hidden = outputs.last_hidden_state
            else:
                outputs = model(
                    input_values,
                    attention_mask=attention_mask,
                    output_hidden_states=True,
                )
                hidden = outputs.hidden_states[layer]

    hidden = hidden.float()
    results: list[dict[str, np.ndarray]] = []

    feat_lengths = None
    if attention_mask is not None:
        lengths = attention_mask.long().sum(dim=1)
        feat_lengths = _get_feat_lengths(model, lengths)

    for b in range(hidden.shape[0]):
        h = hidden[b]
        if attention_mask is not None:
            mask = attention_mask[b].bool()
            if mask.shape[0] != h.shape[0]:
                valid_frames = int(feat_lengths[b].item())
                mask = torch.zeros(h.shape[0], dtype=torch.bool, device=h.device)
                mask[:valid_frames] = True
            valid_h = h[mask]
        else:
            valid_h = h

        if valid_h.numel() == 0:
            valid_h = h

        frames, dim = valid_h.shape
        features: dict[str, np.ndarray] = {
            "ssl_frames": np.asarray(frames, dtype=np.int64),
            "ssl_dim": np.asarray(dim, dtype=np.int64),
        }

        if "mean" in pool:
            mean_vec = (
                valid_h.mean(dim=0)
                .float()
                .cpu()
                .numpy()
                .astype(np.float32, copy=False)
            )
            features["ssl_mean"] = mean_vec
        if "std" in pool:
            std_vec = (
                valid_h.std(dim=0, unbiased=False)
                .float()
                .cpu()
                .numpy()
                .astype(np.float32, copy=False)
            )
            features["ssl_std"] = std_vec

        results.append(features)

    return results


def extract_ssl_embeddings(
    audio: np.ndarray,
    sr: int,
    *,
    model_name: str,
    device: str | None,
    target_sr: int,
    pool: tuple[str, ...],
    layer: int | None,
) -> dict[str, np.ndarray]:
    audio = np.asarray(audio, dtype=np.float32)
    if sr != target_sr:
        audio = _resample(audio, sr, target_sr)
        sr = target_sr

    return extract_ssl_embeddings_batch(
        [audio],
        sr,
        model_name=model_name,
        device=device,
        target_sr=target_sr,
        pool=pool,
        layer=layer,
    )[0]


def flatten_embeddings(prefix: str, vec: np.ndarray) -> dict[str, float]:
    flat: dict[str, float] = {}
    for idx, value in enumerate(vec):
        flat[f"{prefix}_{idx:04d}"] = float(value)
    return flat


def extract_ssl_features(audio: np.ndarray, sr: int) -> dict[str, float]:
    emb = extract_ssl_embeddings(
        audio,
        sr,
        model_name=MODEL_NAME,
        device=DEVICE,
        target_sr=TARGET_SR,
        pool=POOL,
        layer=LAYER,
    )

    features: dict[str, float] = {
        "ssl_frames": float(emb["ssl_frames"]),
        "ssl_dim": float(emb["ssl_dim"]),
    }
    if "ssl_mean" in emb:
        features.update(flatten_embeddings("ssl_mean", emb["ssl_mean"]))
    if "ssl_std" in emb:
        features.update(flatten_embeddings("ssl_std", emb["ssl_std"]))
    return features


In [20]:
df = pd.read_csv(CSV_PATH)  # metadata for paths + labels
df["emotion"] = df["emotion"].astype(str).str.strip().str.lower()

# Filter: valid emotion + agreement > 0
df = df[(df["emotion"] != "xxx") & (df["agreement"] > 0)].copy()
df.shape


(7532, 7)

In [21]:
runtime_device = _detect_device(DEVICE)
runtime_batch_size = _auto_batch_size(runtime_device)
runtime_prefetch_workers = _gpu_prefetch_workers(runtime_device)

if runtime_device == "cuda":
    torch, _ = _require_torch_stack()
    total_vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(
        f"Compute device: {runtime_device} | total_vram={total_vram_gb:.1f} GB | "
        f"batch_size={runtime_batch_size} | prefetch_workers={runtime_prefetch_workers} | amp={USE_AMP}"
    )
else:
    print(
        f"Compute device: {runtime_device} | batch_size={runtime_batch_size} | "
        f"prefetch_workers={runtime_prefetch_workers}"
    )

rows: list[dict[str, float | str | int]] = []
missing: list[str] = []
failed: list[str] = []

processed_paths: set[str] = set()
if OUT_PATH.exists():
    try:
        existing = pd.read_csv(OUT_PATH, usecols=["path"])
        processed_paths = set(existing["path"].astype(str))
    except Exception:
        processed_paths = set()

skipped = len(processed_paths)
if processed_paths:
    df = df[~df["path"].astype(str).isin(processed_paths)].copy()

total = len(df)
if total == 0:
    print("Nothing to process. All paths already extracted.")
else:
    start_time = time.time()
    last_report = 0
    processed = 0
    batches_since_flush = 0
    header_written = OUT_PATH.exists() and OUT_PATH.stat().st_size > 0

    num_batches = (total + runtime_batch_size - 1) // runtime_batch_size
    for batch_idx in tqdm(range(num_batches), desc="Extracting", unit="batch"):
        start = batch_idx * runtime_batch_size
        batch_df = df.iloc[start : start + runtime_batch_size]

        audios: list[np.ndarray] = []
        meta: list[tuple[pd.Series, float, str]] = []

        batch_rows = [row for _, row in batch_df.iterrows()]
        if runtime_prefetch_workers > 1 and len(batch_rows) > 1:
            with ThreadPoolExecutor(
                max_workers=min(runtime_prefetch_workers, len(batch_rows))
            ) as executor:
                loaded_items = list(executor.map(_load_audio_item, batch_rows))
        else:
            loaded_items = [_load_audio_item(row) for row in batch_rows]

        for meta_item, failure, audio in loaded_items:
            if failure is not None:
                if "load_error" in failure:
                    failed.append(failure)
                else:
                    missing.append(failure)
                continue
            if meta_item is None or audio is None:
                continue

            meta.append(meta_item)
            audios.append(audio)

        if not audios:
            continue

        results: list[tuple[tuple[pd.Series, float, str], dict[str, np.ndarray]]] = []
        try:
            embeddings = extract_ssl_embeddings_batch(
                audios,
                TARGET_SR,
                model_name=MODEL_NAME,
                device=runtime_device,
                target_sr=TARGET_SR,
                pool=POOL,
                layer=LAYER,
            )
            for meta_item, emb in zip(meta, embeddings):
                results.append((meta_item, emb))
        except Exception as exc:
            for meta_item, audio in zip(meta, audios):
                try:
                    single = extract_ssl_embeddings_batch(
                        [audio],
                        TARGET_SR,
                        model_name=MODEL_NAME,
                        device=runtime_device,
                        target_sr=TARGET_SR,
                        pool=POOL,
                        layer=LAYER,
                    )[0]
                    results.append((meta_item, single))
                except Exception as inner_exc:
                    rel_path = meta_item[2]
                    failed.append(f"{AUDIO_ROOT / rel_path} | embed_error: {inner_exc}")

        if not results:
            continue

        for (row, duration_s, rel_path), emb in results:
            features: dict[str, float] = {
                "ssl_frames": float(emb["ssl_frames"]),
                "ssl_dim": float(emb["ssl_dim"]),
            }
            if "ssl_mean" in emb:
                features.update(flatten_embeddings("ssl_mean", emb["ssl_mean"]))
            if "ssl_std" in emb:
                features.update(flatten_embeddings("ssl_std", emb["ssl_std"]))

            record: dict[str, float | str | int] = {
                "path": str(rel_path),
                "session": int(row["session"]),
                "method": row["method"],
                "gender": row["gender"],
                "emotion": row["emotion"],
                "n_annotators": int(row["n_annotators"]),
                "agreement": int(row["agreement"]),
                "duration_s": float(duration_s),
            }
            record.update(features)
            rows.append(record)
            processed += 1

        batches_since_flush += 1
        if batches_since_flush >= FLUSH_EVERY:
            if rows:
                pd.DataFrame(rows).to_csv(
                    OUT_PATH,
                    mode="a",
                    index=False,
                    header=not header_written,
                )
                header_written = True
                rows.clear()
            batches_since_flush = 0

        if processed - last_report >= 100:
            elapsed = max(1e-6, time.time() - start_time)
            avg_sec = elapsed / max(1, processed)
            utt_per_sec = processed / elapsed
            remaining = max(0, total - processed)
            eta_s = remaining * avg_sec
            print(
                f"Processed {processed}/{total} (skipped {skipped}) | "
                f"{utt_per_sec:.2f} utt/s | avg {avg_sec:.2f}s/utt | ETA ~{eta_s/60:.1f}m"
            )
            last_report = processed

    if rows:
        pd.DataFrame(rows).to_csv(
            OUT_PATH,
            mode="a",
            index=False,
            header=not header_written,
        )

print(f"Saved: {OUT_PATH}")
if missing:
    print(f"Missing audio files: {len(missing)}")
if failed:
    print(f"Failed files: {len(failed)}")
    failed_path = OUT_DIR / "ssl_failed.txt"
    failed_path.write_text("\n".join(failed), encoding="utf-8")
    print(f"Failed list: {failed_path}")


Extracting:   2%|▏         | 15/942 [00:03<01:46,  8.74batch/s]

Processed 104/7532 (skipped 0) | 30.50 utt/s | avg 0.03s/utt | ETA ~4.1m


Extracting:   3%|▎         | 27/942 [00:05<01:57,  7.78batch/s]

Processed 208/7532 (skipped 0) | 37.83 utt/s | avg 0.03s/utt | ETA ~3.2m


Extracting:   4%|▍         | 39/942 [00:07<02:41,  5.59batch/s]

...

Processed 2600/7532 (skipped 0) | 48.22 utt/s | avg 0.02s/utt | ETA ~1.7m


Extracting:  36%|███▌      | 339/942 [00:56<01:30,  6.66batch/s]

Processed 2704/7532 (skipped 0) | 48.26 utt/s | avg 0.02s/utt | ETA ~1.7m


Extracting:  36%|███▌      | 341/942 [00:56<01:39,  6.03batch/s]


KeyboardInterrupt: 